In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
# ███████╗███╗   ███╗ █████╗ ██████╗ ████████╗      █████╗ ██████╗ ███████╗
# ██╔════╝████╗ ████║██╔══██╗██╔══██╗╚══██╔══╝     ██╔══██╗██╔══██╗██╔════╝
# ███████╗██╔████╔██║███████║██████╔╝   ██║        ███████║██████╔╝███████╗
# ╚════██║██║╚██╔╝██║██╔══██║██╔══██╗   ██║        ██╔══██║██╔═══╝ ╚════██║
# ███████║██║ ╚═╝ ██║██║  ██║██║  ██║   ██║        ██║  ██║██║     ███████║
# ╚══════╝╚═╝     ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝        ╚═╝  ╚═╝╚═╝     ╚══════╝
#
#  Smart APS V7  —  Level 2 Intelligence Upgrade
#
#  What's new vs V6:
#  ─────────────────────────────────────────────────────────────
#  1. PRIORITY SCORING ENGINE
#     Every active part gets a dynamic score (0–100) each run:
#       score = (urgency_weight × days_gap_score)
#             + (category_weight × category_tier_score)
#             + (indent_weight × relative_indent_score)
#     Parts are scheduled in score order — highest score first.
#     A Runner with zero inventory always scores highest.
#
#  2. SMART QTY ENGINE  (replaces fixed MAX_OVERPRODUCTION_DAYS)
#     qty_to_produce is now decided per-part per-scenario:
#       Scenario 0 (all critical)   → survive today only — every
#                                     hour counts, don't over-produce
#       Scenario 1 (some critical)  → critical parts: n days where
#                                     n = model decides using avail hrs
#                                     non-critical: defer if possible
#       Scenario 2 (below buffer)   → produce to reach 3-day target
#       Scenario 3 (all healthy)    → cycle parts, produce by mean
#                                     daily indent, rotate coverage
#
#  3. CROSS-MACHINE REBALANCING  (Uber-style routing)
#     When a part cannot fit on its primary (best-ranked) machine:
#       Pass A: try every other compatible machine in priority order
#       Pass B: if all machines are full for this part, check if any
#               lower-priority part already planned on a machine can
#               have its hours trimmed to free space (trim-and-insert)
#       Pass C: split the part across two compatible machines if
#               single-machine capacity is exhausted but combined
#               capacity is sufficient  (split-run)
#     Result: far fewer "NOT PLANNED" parts due to capacity.
#
#  4. 22H UTILIZATION ENFORCER
#     After primary scheduling, a dedicated filler pass runs PER
#     MACHINE in utilization order (least-used first):
#       Step 1: Can we extend an already-planned part on this machine?
#               → yes, if the part is below its scenario-target qty
#                 AND extending doesn't violate the overproduction cap
#       Step 2: Can we add a new part (≥ MIN_RUN_HOURS free)?
#               → yes, sorted by priority score descending
#               → changeover cost subtracted from available run time
#       Step 3: If remaining < MIN_RUN_HOURS and > 0.25h, log as
#               "micro-idle" — too small to fill but tracked
#     Every machine is expected to reach ≥ 98% utilization unless
#     truly all compatible parts are either planned or capped.
#
#  5. MACHINE STATE  (unchanged from V6 — still from JSON)
#     Today's plan saves state → tomorrow's run reads it for free
#     changeover decisions.
#
#  Planning logic (unchanged from V6):
#    daily_indent  = monthly_indent / working_days
#    today_target  = max(0, daily_indent − inventory)
#    working_days  = calendar days − Sundays
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22        # machine hours per shift
MIN_RUN_HOURS        = 4         # minimum viable run block
TARGET_DAYS_INV      = 3         # ideal inventory buffer (days)
MACHINE_STATE_FILE   = "machine_state.json"

# Skip thresholds — absolute, no exceptions
MIN_DAILY_INDENT     = 150       # skip if daily_indent ≤ this
MIN_INDENT_HOURS     = 4.0       # skip if whole monthly indent ≤ this many hours

# Overproduction cap per scenario (days of daily_indent)
# These replace the single MAX_OVERPRODUCTION_DAYS from V6.
# The model picks the right cap based on the scenario.
OPD_SCENARIO_0 = 1.5   # critical — produce just enough to survive + small buffer
OPD_SCENARIO_1 = 3.0   # mixed    — critical parts get 3 days; others defer
OPD_SCENARIO_2 = 4.0   # below target — fill up to 4-day supply
OPD_SCENARIO_3 = 5.0   # healthy — normal ceiling, cycle parts evenly

# Priority score weights (must sum to 1.0)
W_URGENCY  = 0.55   # days_of_inventory gap vs target — highest weight
W_CATEGORY = 0.25   # Runner > Repeater > Stranger
W_INDENT   = 0.20   # relative daily_indent among peers

# Utilization target — machines below this are flagged UNDERUSED
UTIL_TARGET_PCT = 98.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V7_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V7  —  Level 2 Intelligence  —  VT Only")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days   : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Util target    : {UTIL_TARGET_PCT}%  per machine")
print(f"  Priority weights: urgency={W_URGENCY}  category={W_CATEGORY}  indent={W_INDENT}")
print(f"  OPD caps by scenario: S0={OPD_SCENARIO_0}d  S1={OPD_SCENARIO_1}d  "
      f"S2={OPD_SCENARIO_2}d  S3={OPD_SCENARIO_3}d")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["_cv"] = pd.to_numeric(data[vt_col_cavity],    errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")
if len(data_zero_rate) > 0:
    print(f"  Zero/missing cycle time : {len(data_zero_rate)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — SKIP RULES  (absolute, no exceptions)
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    return False, ""

# =============================================================
# SECTION 7B — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover        = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

print(f"  Changeover times loaded : {len(vt_changeover)} machines")

# =============================================================
# SECTION 7C — PART CATEGORY
# =============================================================

def build_category(df, sheet_name):
    cat = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category = build_category(vt_parts_raw, "VT")
CATEGORY_TIER = {"Runner": 0, "Repeater": 1, "Stranger": 2}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"\n  Machine state loaded  ({len(state)} machines with history)")
        return state
    print(f"\n  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(vt_state):
    combined = {m: p for m, p in vt_state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day indent)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{TARGET_DAYS_INV} days inventory)"

# =============================================================
# SECTION 11 — OVERPRODUCTION CAP BY SCENARIO
# =============================================================

def opd_cap(scenario_id):
    """Return the overproduction days cap for the given scenario."""
    return {0: OPD_SCENARIO_0,
            1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2,
            3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)

# =============================================================
# SECTION 12 — PRIORITY SCORING ENGINE  (NEW in V7)
# =============================================================
#
# Score = W_URGENCY × urgency_score
#       + W_CATEGORY × category_score
#       + W_INDENT   × indent_score
#
# urgency_score  (0–100): how far below the TARGET_DAYS_INV buffer
#   the part is. Zero inventory = 100. At exactly target = 0.
#   Parts above target get a negative urgency (de-prioritised).
#
# category_score (0–100): Runner=100, Repeater=60, Stranger=20
#
# indent_score   (0–100): normalised relative to the max daily_indent
#   across all active parts. Highest demand = 100.
#
# =============================================================

CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

def compute_priority_scores(active_parts):
    """
    Returns dict {part: score} for all active parts.
    Also returns the raw components for the audit sheet.
    """
    # Collect raw values
    rows = []
    for p in active_parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        cat   = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0

        # Urgency: how many days below TARGET_DAYS_INV
        # If inv=0 → gap = TARGET_DAYS_INV → max urgency
        # If inv >= TARGET_DAYS_INV × daily → gap = 0 or negative
        gap_days = max(0.0, TARGET_DAYS_INV - days_cov)
        urgency_raw = gap_days / TARGET_DAYS_INV  # 0..1 (capped at 1)
        urgency_raw = min(1.0, urgency_raw)

        rows.append({
            "part":        p,
            "inv":         inv,
            "daily":       daily,
            "days_cov":    days_cov,
            "cat":         cat,
            "urgency_raw": urgency_raw,
        })

    if not rows:
        return {}, []

    # Normalise indent_score within this group
    max_daily = max(r["daily"] for r in rows) or 1.0

    scores = {}
    score_rows = []
    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100

        final_score = (
            W_URGENCY  * urgency_score  +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )

        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — SMART QTY ENGINE  (NEW in V7)
# =============================================================
#
# Determines how many hours to run a part given:
#   • scenario_id     — 0/1/2/3
#   • current_inv     — live inventory (updated as plan builds)
#   • daily_indent    — daily requirement
#   • rate            — units per hour
#   • available_run   — hours remaining on chosen machine after CO
#
# Returns target_run_hrs — the number of hours to schedule.
#
# Logic:
#   1. Compute headroom = opd_cap × daily − current_inv
#      (how many units we are allowed to produce before hitting cap)
#   2. hrs_to_fill_target = (TARGET_DAYS_INV × daily − current_inv) / rate
#      (hours needed to reach the 3-day buffer target)
#   3. Scenario adjustment:
#      S0: run only until today's demand is met + 20% margin
#          (conserve machine time for other critical parts)
#      S1: critical part → run to fill 3-day target if hours allow
#          (non-critical parts are deferred before reaching this fn)
#      S2: run to fill 3-day target
#      S3: run by daily mean (1 day of supply per scheduling cycle,
#          so all parts get turns across days)
#   4. Apply floors and ceilings:
#      floor : MIN_RUN_HOURS
#      ceiling: min(available_run, hrs_to_cap)
#
# =============================================================

def smart_target_hours(part, scenario_id, current_inv, daily, r_val, available_run):
    """
    Returns (target_run_hrs, reasoning_str).
    """
    if daily <= 0 or r_val <= 0:
        return MIN_RUN_HOURS, "no demand data — fallback to MIN_RUN_HOURS"

    cap_days      = opd_cap(scenario_id)
    headroom_qty  = max(0.0, cap_days * daily - current_inv)
    hrs_to_cap    = headroom_qty / r_val

    # Hours to reach 3-day buffer from current inventory
    shortfall_qty = max(0.0, TARGET_DAYS_INV * daily - current_inv)
    hrs_to_target = shortfall_qty / r_val

    if scenario_id == 0:
        # All critical — produce just enough to survive today
        # + 50% margin, then stop and give time to next part
        survive_qty   = max(0.0, daily - current_inv)
        hrs_survive   = survive_qty / r_val
        target_hrs    = hrs_survive * 1.5
        reason        = f"S0: survive today ({hrs_survive:.2f}h) × 1.5 margin"

    elif scenario_id == 1:
        # This part is here because it's critical (scored high).
        # Produce to fill 3-day buffer if machine allows.
        target_hrs = hrs_to_target
        reason     = f"S1: fill to {TARGET_DAYS_INV}-day buffer ({target_hrs:.2f}h needed)"

    elif scenario_id == 2:
        # Below buffer — fill to 3-day target
        target_hrs = hrs_to_target
        reason     = f"S2: fill to {TARGET_DAYS_INV}-day buffer ({target_hrs:.2f}h needed)"

    else:  # scenario 3 — all healthy
        # Produce 1 day of supply this cycle; rotate tomorrow
        target_hrs = daily / r_val
        reason     = f"S3: 1-day cycle quantity ({target_hrs:.2f}h)"

    # Apply floor and ceiling
    target_hrs = max(MIN_RUN_HOURS, target_hrs)
    target_hrs = min(target_hrs, hrs_to_cap)
    target_hrs = min(target_hrs, available_run)
    target_hrs = max(target_hrs, MIN_RUN_HOURS)     # re-apply after ceiling

    return round(target_hrs, 3), reason

# =============================================================
# SECTION 14 — MACHINE RANKER  (enhanced in V7)
# =============================================================
#
# Ranks compatible machines for a given part.
# Returns list of (machine, co_hrs, effective_free_hrs, rank_cost).
#
# Ranking cost = f(changeover, utilization, last_part_match).
# A machine where the same part ran last is strongly preferred
# (co_hrs = 0, cost penalty removed).
# A heavily-used machine is penalised (less spare time = higher cost).
#
# Runner lock: a Runner with inv ≤ 1 day MUST stay on the same
# machine it last ran. If no such machine exists → not planned.
#
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, changeover_dict, inv_days):
    """
    Returns (ranked_list, runner_lock).
    ranked_list: [(machine, co_hrs, effective_free, cost), ...]
    """
    category   = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue

        last = machine_last_part.get(m)  # None = first run

        # Runner lock: only the machine that last ran this part
        if runner_lock and last != part:
            continue

        # Changeover cost
        if last is None or last == part:
            co_hrs = 0.0
        else:
            co_hrs = changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)

        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        # Rank cost: lower is better
        # Prefer: same last part (co=0) + machine with most free time
        co_penalty      = co_hrs / AVAILABLE_HOURS
        util_penalty    = used / AVAILABLE_HOURS
        same_part_bonus = -0.20 if (last == part) else 0.0
        cost = co_penalty + util_penalty + same_part_bonus

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock

# =============================================================
# SECTION 14B — TRIM-AND-INSERT  (NEW in V7)
# =============================================================
#
# When a high-priority part cannot be planned because all compatible
# machines are full, we attempt to free space by trimming a lower-
# priority part already planned on one of those machines.
#
# Rules:
#   • Only trim if the trimmed part still has inv_after ≥ daily_indent
#     (i.e. inventory + remaining_production covers today's need)
#   • Trim the lowest-score planned part on the target machine.
#   • Maximum trim: to MIN_RUN_HOURS (never trim a part below that).
#   • The freed time must be ≥ MIN_RUN_HOURS after subtracting CO.
#
# Returns (machine, freed_hrs, co_hrs) or None if not possible.
#
# =============================================================

def try_trim_and_insert(part_to_insert, plan, machine_hours,
                        machine_last_part, current_inventory,
                        priority_scores, scenario_id):
    """
    Try to free machine time for part_to_insert by trimming a
    lower-priority planned part.
    Returns (machine, freed_run_hrs, co_hrs) or None.
    """
    compatible = vt_compat.get(part_to_insert, [])
    inv_new    = current_inventory.get(part_to_insert, 0)
    daily_new  = indent_daily.get(part_to_insert, 0)
    r_new      = rate.get(part_to_insert, 1)
    score_new  = priority_scores.get(part_to_insert, 0)

    best_option = None  # (machine, freed, co_hrs, trim_amount, row_to_trim)

    for m in compatible:
        # Parts already planned on this machine
        m_rows = [row for row in plan if row["Machine"] == m]
        if not m_rows:
            continue

        for row in m_rows:
            p_existing = row["Part"]
            score_existing = priority_scores.get(p_existing, 0)

            # Only trim a part with LOWER priority score
            if score_existing >= score_new:
                continue

            run_h_current = float(row.get("Run_Hours", 0))
            r_existing    = float(row.get("Rate_Per_Hour", rate.get(p_existing, 1)))
            inv_existing  = inventory.get(p_existing, 0)
            produced_now  = float(row.get("Production_Qty", 0))
            daily_ex      = indent_daily.get(p_existing, 0)

            # How much can we trim?
            # Must leave: MIN_RUN_HOURS for the existing part
            # Must ensure: remaining_production + inv ≥ daily_indent
            max_trim_hrs = run_h_current - MIN_RUN_HOURS
            if max_trim_hrs < 0.25:
                continue

            # Check if trimmed production still covers today's demand
            # Trim the max we can and see if demand is still met
            min_qty_needed = max(0.0, daily_ex - inv_existing)
            min_hrs_needed = min_qty_needed / r_existing if r_existing > 0 else MIN_RUN_HOURS
            min_hrs_needed = max(MIN_RUN_HOURS, min_hrs_needed)

            trim_amount = run_h_current - min_hrs_needed
            trim_amount = min(trim_amount, max_trim_hrs)
            if trim_amount < 0.25:
                continue

            # Check if freed time is enough for part_to_insert
            last  = machine_last_part.get(m)
            co_hr = 0.0 if (last is None or last == p_existing) else 0.0
            # CO for the NEW part after existing part
            co_for_new = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            if p_existing == part_to_insert:
                co_for_new = 0.0

            freed_run = trim_amount - co_for_new
            if freed_run < MIN_RUN_HOURS:
                continue

            # Valid option — pick the one that frees the most time
            if best_option is None or freed_run > best_option[1]:
                best_option = (m, freed_run, co_for_new, trim_amount, row)

    if best_option is None:
        return None

    m, freed_run, co_for_new, trim_amount, row_to_trim = best_option

    # Apply the trim
    old_run       = float(row_to_trim["Run_Hours"])
    new_run       = round(old_run - trim_amount, 3)
    r_trim        = float(row_to_trim.get("Rate_Per_Hour", 1))
    lost_qty      = round(trim_amount * r_trim, 0)

    row_to_trim["Run_Hours"]      = new_run
    row_to_trim["Production_Qty"] = round(float(row_to_trim.get("Production_Qty", 0)) - lost_qty, 0)
    row_to_trim["Total_Hrs_Used"] = round(float(row_to_trim.get("Changeover_Hrs", 0)) + new_run, 3)
    row_to_trim["Type"]           = row_to_trim.get("Type", "Primary") + "→TRIMMED"
    row_to_trim["Trim_Note"]      = (f"Trimmed {trim_amount:.2f}h to make room for "
                                     f"{part_to_insert} (higher priority)")

    # Update machine hours
    machine_hours[m] = round(machine_hours.get(m, 0) - trim_amount, 4)
    p_trimmed = row_to_trim["Part"]
    current_inventory[p_trimmed] = max(0.0,
        current_inventory.get(p_trimmed, 0) - lost_qty)

    print(f"    ✂  TRIM {p_trimmed:28s} on {m:15s}  "
          f"−{trim_amount:.2f}h  freed {freed_run:.2f}h for {part_to_insert}")

    return (m, freed_run, co_for_new)

# =============================================================
# SECTION 15 — SPLIT-RUN  (NEW in V7)
# =============================================================
#
# When no single machine has enough free time for a part but
# TWO compatible machines together do, we split the production:
#   • Machine A runs until its capacity is exhausted
#   • Machine B runs the remainder
#
# Rules:
#   • Both machines must have ≥ MIN_RUN_HOURS free (after CO).
#   • Total production across both = smart_target_hours qty.
#   • The split is noted in the plan with Type = "Split-Run A/B".
#
# Returns list of (machine, co_hrs, run_hrs) or empty list.
#
# =============================================================

def try_split_run(part, scenario_id, machine_hours, machine_last_part,
                  current_inventory, changeover_dict):
    """
    Returns [(m1, co1, run1), (m2, co2, run2)] or [].
    """
    daily = indent_daily.get(part, 0)
    r_val = rate.get(part, 1)
    inv   = current_inventory.get(part, 0)
    inv_days = inv / daily if daily > 0 else 999

    compatible = vt_compat.get(part, [])
    ranked, _ = rank_machines(part, compatible, machine_hours,
                               machine_last_part, changeover_dict, inv_days)

    # Need at least two viable machines
    viable = [(m, co, eff, cost) for m, co, eff, cost in ranked
              if eff >= MIN_RUN_HOURS]
    if len(viable) < 2:
        return []

    # Determine total target hours
    # Use max free time from first machine, remainder from second
    m1, co1, eff1, _ = viable[0]
    m2, co2, eff2, _ = viable[1]

    cap_days     = opd_cap(scenario_id)
    headroom_qty = max(0.0, cap_days * daily - inv)
    total_target = headroom_qty / r_val if r_val > 0 else MIN_RUN_HOURS

    run1 = min(eff1, total_target)
    run1 = max(run1, MIN_RUN_HOURS)

    remaining_qty = max(0.0, total_target * r_val - run1 * r_val)
    run2 = remaining_qty / r_val if r_val > 0 else 0
    run2 = max(run2, MIN_RUN_HOURS)
    run2 = min(run2, eff2)

    if run2 < MIN_RUN_HOURS:
        return []

    return [(m1, co1, round(run1, 3)), (m2, co2, round(run2, 3))]

# =============================================================
# SECTION 16 — CSP TOOL-CHANGER  (unchanged from V6)
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"


def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events


def _recompute_natural_start(ev, plan):
    m      = ev["machine"]
    target = ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor


def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)


def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra


def _csp_order_events(events):
    n = len(events)
    if n == 0:
        return events
    try:
        from constraint import Problem
        SLOTS = int(AVAILABLE_HOURS * 60)
        problem = Problem()
        for i, ev in enumerate(events):
            dur_min = int(math.ceil(ev["co_duration"] * 60))
            problem.addVariable(i, range(0, max(0, SLOTS - dur_min) + 1))
        for i in range(n):
            for j in range(i+1, n):
                di = int(math.ceil(events[i]["co_duration"] * 60))
                dj = int(math.ceil(events[j]["co_duration"] * 60))
                def make_c(di, dj):
                    return lambda si, sj: (si+di <= sj) or (sj+dj <= si)
                problem.addConstraint(make_c(di, dj), (i, j))
        solutions = problem.getSolutions()
        if not solutions:
            return sorted(events, key=lambda e: e["natural_start"])
        best = min(solutions, key=lambda sol: sum(sol.values()))
        ordered = sorted(range(n), key=lambda i: best[i])
        return [events[i] for i in ordered]
    except ImportError:
        return sorted(events, key=lambda e: e["natural_start"])
    except Exception:
        return sorted(events, key=lambda e: e["natural_start"])


def stagger_changeovers(plan, machines, changeover_dict):
    print(f"\n  CSP Tool-Changer Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print("  No changeovers — tool changer idle  ✓")
        return

    ordered_events = _csp_order_events(events)
    tool_changer_free_at = 0.0
    total_extra_pcs      = 0

    print(f"\n  {'#':<4} {'Machine':<18} {'Part Before':<24} {'Part After':<24} "
          f"{'CO':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7}")
    print(f"  {'─'*4} {'─'*18} {'─'*24} {'─'*24} "
          f"{'─'*5} {'─'*8} {'─'*8} {'─'*7}")

    for idx, ev in enumerate(ordered_events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = actual_start - natural_start

        if wait_hrs > 0.001:
            extended_hrs, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs

        ev["actual_start"] = actual_start
        tool_changer_free_at = actual_start + co_h

        wait_str = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<24} "
              f"{ev['part_after']:<24} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7}")

    print(f"\n  Tool-changer finishes at {_fmt_h(tool_changer_free_at)}  |  "
          f"Extra pcs from wait-fill: {total_extra_pcs:,.0f}")

# =============================================================
# SECTION 17 — 22H UTILIZATION ENFORCER  (NEW in V7)
# =============================================================
#
# Runs AFTER primary scheduling. Processes machines in order of
# ascending utilization (least-used first — they need the most help).
#
# For each under-utilized machine:
#   STEP 1: EXTEND existing parts
#     For each planned part on this machine, sorted by priority score:
#       • Compute headroom = opd_cap × daily − current_inventory
#       • Extend run by min(remaining_free, headroom/rate)
#       • Continue until machine hits AVAILABLE_HOURS or no headroom
#
#   STEP 2: ADD new parts
#     Candidates = parts not yet planned + compatible with this machine
#     Sorted by priority score descending
#     For each candidate:
#       • Compute CO cost
#       • effective_free = remaining − co
#       • If effective_free ≥ MIN_RUN_HOURS: plan it
#       • Use smart_target_hours for qty
#       • Continue until machine is full or no more candidates
#
#   STEP 3: LOG micro-idle
#     If remaining > 0 after both steps: log as micro-idle
#
# =============================================================

def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER")
    print(f"  Target: ≥{UTIL_TARGET_PCT}% on every machine")
    print(f"  Scenario: {scenario_id}  |  OPD cap: {opd_cap(scenario_id)} days")

    # Process machines least-utilized first
    machines_by_util = sorted(
        vt_machines,
        key=lambda m: machine_hours.get(m, 0)
    )

    micro_idle_log = []

    for m in machines_by_util:
        used      = machine_hours.get(m, 0)
        remaining = round(AVAILABLE_HOURS - used, 4)
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)

        if remaining < 0.05:
            continue  # machine is full

        # ── STEP 1: Extend existing parts ────────────────────
        parts_on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0),
                        reverse=True):
            if remaining < 0.05:
                break
            daily_p   = indent_daily.get(p, 0)
            r_val     = rate.get(p, 1)
            inv_now   = current_inventory.get(p, 0)
            cap       = opd_cap(scenario_id) * daily_p
            headroom  = max(0.0, cap - inv_now)
            extend_by = min(remaining, headroom / r_val if r_val > 0 else 0)

            if extend_by < 0.05:
                continue

            extra_qty = round(extend_by * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + extend_by, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break

            machine_hours[m]    = round(machine_hours.get(m, 0) + extend_by, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining             = round(remaining - extend_by, 4)

            print(f"    ↑ EXTEND {p:28s} on {m:15s}  "
                  f"+{extend_by:.2f}h  qty+={extra_qty:.0f}")

        # ── STEP 2: Add new parts ─────────────────────────────
        if remaining < MIN_RUN_HOURS:
            continue

        # Build candidate list
        candidates = []
        for p in all_parts:
            if p in already_planned:
                continue
            if m not in vt_compat.get(p, []):
                continue
            skip, _ = should_skip(p)
            if skip:
                continue
            daily_p  = indent_daily.get(p, 0)
            inv_now  = current_inventory.get(p, 0)
            cap      = opd_cap(scenario_id) * daily_p
            if inv_now >= cap:
                continue  # already at cap — no point producing more
            candidates.append(p)

        # Sort by priority score
        candidates.sort(key=lambda x: priority_scores.get(x, 0), reverse=True)

        for p in candidates:
            if remaining < MIN_RUN_HOURS:
                break

            daily_p = indent_daily.get(p, 0)
            r_val   = rate.get(p, 1)
            inv_now = current_inventory.get(p, 0)
            inv_days = inv_now / daily_p if daily_p > 0 else 999

            last   = machine_last_part.get(m)
            co_hrs = 0.0 if (last is None or last == p) else \
                     vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)

            effective_free = round(remaining - co_hrs, 4)
            if effective_free < MIN_RUN_HOURS:
                continue

            target_hrs, reason = smart_target_hours(
                p, scenario_id, inv_now, daily_p, r_val, effective_free)

            qty = round(target_hrs * r_val, 0)

            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + target_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            already_planned.add(p)
            remaining = round(remaining - co_hrs - target_hrs, 4)

            plan.append({
                "Part":              p,
                "Category":          part_category.get(p, "Stranger"),
                "Machine":           m,
                "Run_Hours":         round(target_hrs, 3),
                "Changeover_Hrs":    round(co_hrs, 3),
                "Total_Hrs_Used":    round(co_hrs + target_hrs, 3),
                "Rate_Per_Hour":     round(r_val, 2),
                "Production_Qty":    qty,
                "Monthly_Indent":    round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":      round(daily_p, 2),
                "Today_Target":      round(today_target_qty.get(p, 0), 0),
                "Changeover":        "No" if co_hrs == 0 else "Yes",
                "Type":              "Filler (utilization enforcer)",
                "Runner_Lock":       "No",
                "Priority_Score":    round(priority_scores.get(p, 0), 2),
                "Qty_Reason":        reason,
                "Stagger_Adjusted":  "No",
            })
            co_str = "No CO" if co_hrs == 0 else f"CO {co_hrs*60:.0f}min"
            print(f"    + ADD  {p:28s} → {m:15s}  "
                  f"{target_hrs:.2f}h  qty={qty:.0f}  [{co_str}]  {reason[:40]}")

        # ── STEP 3: Micro-idle log ────────────────────────────
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":          m,
                    "Idle_Hrs":         round(final_remaining, 3),
                    "Utilization_Pct":  util_final,
                    "Note": ("All compatible parts either fully planned or at OPD cap — "
                             "true capacity gap")
                })
                print(f"    ⚠  {m:15s}  micro-idle {final_remaining:.2f}h  "
                      f"({util_final}%) — no more eligible parts")

    return micro_idle_log

# =============================================================
# SECTION 18 — INDENT HORIZON TABLE
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 19 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, changeover_dict, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*65}")

    # ── Classify scenario ─────────────────────────────────────
    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # ── Active parts (not skipped AND inventory insufficient) ─
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and not (inventory.get(p, 0) >= indent_daily.get(p, 0) > 0)
    ]

    # ── Priority scores ───────────────────────────────────────
    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    print(f"\n  Priority scores computed for {len(active_parts)} active parts:")
    if score_rows:
        sorted_scores = sorted(score_rows, key=lambda r: r["Final_Score"], reverse=True)
        print(f"  {'Part':<30}  {'Cat':<10}  {'Score':>7}  {'Urgency':>7}  "
              f"{'Inv Days':>8}  {'Daily':>7}")
        print(f"  {'─'*30}  {'─'*10}  {'─'*7}  {'─'*7}  {'─'*8}  {'─'*7}")
        for r in sorted_scores[:20]:   # show top 20
            print(f"  {r['Part']:<30}  {r['Category']:<10}  "
                  f"{r['Final_Score']:>7.1f}  "
                  f"{r['Urgency_Score']:>7.1f}  "
                  f"{r['Days_Coverage']:>8.2f}  "
                  f"{r['Daily_Indent']:>7.2f}")
        if len(sorted_scores) > 20:
            print(f"  ... ({len(sorted_scores)-20} more)")

    # ── State tracking ────────────────────────────────────────
    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned = [], [], []
    already_planned = set()

    # ── Sort active parts by priority score descending ────────
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0),
                           reverse=True)

    print(f"\n  PRIMARY SCHEDULING PASS (score-ordered):")
    print(f"  {'Part':<30}  {'Score':>6}  {'Machine':<15}  {'Run':>5}  "
          f"{'Qty':>8}  {'CO':>4}  Status")
    print(f"  {'─'*30}  {'─'*6}  {'─'*15}  {'─'*5}  {'─'*8}  {'─'*4}  {'─'*20}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        r_val    = rate.get(part, 1)
        category = part_category.get(part, "Stranger")
        inv_days = inv_now / daily if daily > 0 else 999
        score    = priority_scores.get(part, 0)

        compatible_mch = compatibility.get(part, [])

        if monthly == 0:
            deferred.append({
                "Part": part, "Category": category,
                "Reason": "Monthly indent = 0",
                "Next_Action": "Set indent in VT sheet",
            })
            print(f"  {part:<30}  {score:>6.1f}  {'—':<15}  {'—':>5}  "
                  f"{'—':>8}  {'—':>4}  DEFERRED (no indent)")
            continue

        if not compatible_mch:
            not_planned.append({
                "Part": part, "Category": category,
                "Score": score,
                "Daily_Indent": round(daily, 2),
                "Inventory_Now": round(inv_now, 0),
                "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix",
                "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30}  {score:>6.1f}  {'—':<15}  {'—':>5}  "
                  f"{'—':>8}  {'—':>4}  NOT PLANNED — not in matrix")
            continue

        # ── Rank machines — Pass A ────────────────────────────
        ranked, runner_lock = rank_machines(
            part, compatible_mch, machine_hours,
            machine_last_part, changeover_dict, inv_days)

        assigned = False

        for m, co_hrs, effective_free, cost in ranked:
            target_hrs, reason = smart_target_hours(
                part, scenario_id, inv_now, daily, r_val, effective_free)

            qty = round(target_hrs * r_val, 0)
            machine_hours[m]       += co_hrs + target_hrs
            current_inventory[part] = current_inventory.get(part, 0) + qty
            machine_last_part[m]    = part
            already_planned.add(part)

            plan.append({
                "Part":              part,
                "Category":          category,
                "Machine":           m,
                "Run_Hours":         round(target_hrs, 3),
                "Changeover_Hrs":    round(co_hrs, 3),
                "Total_Hrs_Used":    round(co_hrs + target_hrs, 3),
                "Rate_Per_Hour":     round(r_val, 2),
                "Production_Qty":    qty,
                "Monthly_Indent":    round(monthly, 0),
                "Daily_Indent":      round(daily, 2),
                "Today_Target":      round(today_target_qty.get(part, 0), 0),
                "Changeover":        "No" if co_hrs == 0 else "Yes",
                "Type":              "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
                "Runner_Lock":       "YES" if runner_lock else "No",
                "Priority_Score":    score,
                "Qty_Reason":        reason,
                "Stagger_Adjusted":  "No",
            })
            co_str = "No" if co_hrs == 0 else f"Yes"
            flag   = " [ZERO-INV]" if inv_now == 0 else \
                     " [RUNNER-LOCK]" if runner_lock else ""
            print(f"  {part:<30}  {score:>6.1f}  {m:<15}  "
                  f"{target_hrs:>5.2f}  {qty:>8.0f}  {co_str:>4}  "
                  f"✓{flag}")
            assigned = True
            break

        if assigned:
            continue

        # ── Pass B: Trim-and-insert ───────────────────────────
        trim_result = try_trim_and_insert(
            part, plan, machine_hours, machine_last_part,
            current_inventory, priority_scores, scenario_id)

        if trim_result:
            m, freed_run, co_for_new = trim_result
            target_hrs, reason = smart_target_hours(
                part, scenario_id, inv_now, daily, r_val, freed_run)
            qty = round(target_hrs * r_val, 0)

            machine_hours[m]       += co_for_new + target_hrs
            current_inventory[part] = current_inventory.get(part, 0) + qty
            machine_last_part[m]    = part
            already_planned.add(part)

            plan.append({
                "Part":              part,
                "Category":          category,
                "Machine":           m,
                "Run_Hours":         round(target_hrs, 3),
                "Changeover_Hrs":    round(co_for_new, 3),
                "Total_Hrs_Used":    round(co_for_new + target_hrs, 3),
                "Rate_Per_Hour":     round(r_val, 2),
                "Production_Qty":    qty,
                "Monthly_Indent":    round(monthly, 0),
                "Daily_Indent":      round(daily, 2),
                "Today_Target":      round(today_target_qty.get(part, 0), 0),
                "Changeover":        "No" if co_for_new == 0 else "Yes",
                "Type":              "Primary (after trim)",
                "Runner_Lock":       "No",
                "Priority_Score":    score,
                "Qty_Reason":        reason + " [trim-and-insert]",
                "Stagger_Adjusted":  "No",
            })
            print(f"  {part:<30}  {score:>6.1f}  {m:<15}  "
                  f"{target_hrs:>5.2f}  {qty:>8.0f}  "
                  f"{'Yes' if co_for_new>0 else 'No':>4}  ✓ [TRIM-INSERT]")
            continue

        # ── Pass C: Split-run ─────────────────────────────────
        split = try_split_run(part, scenario_id, machine_hours,
                              machine_last_part, current_inventory,
                              changeover_dict)
        if split:
            for idx, (m, co_hrs, run_h) in enumerate(split):
                qty = round(run_h * r_val, 0)
                machine_hours[m]       += co_hrs + run_h
                current_inventory[part] = current_inventory.get(part, 0) + qty
                machine_last_part[m]    = part
                split_label = f"Split-Run {'A' if idx==0 else 'B'}"
                plan.append({
                    "Part":              part,
                    "Category":          category,
                    "Machine":           m,
                    "Run_Hours":         run_h,
                    "Changeover_Hrs":    round(co_hrs, 3),
                    "Total_Hrs_Used":    round(co_hrs + run_h, 3),
                    "Rate_Per_Hour":     round(r_val, 2),
                    "Production_Qty":    qty,
                    "Monthly_Indent":    round(monthly, 0),
                    "Daily_Indent":      round(daily, 2),
                    "Today_Target":      round(today_target_qty.get(part, 0), 0),
                    "Changeover":        "No" if co_hrs == 0 else "Yes",
                    "Type":              split_label,
                    "Runner_Lock":       "No",
                    "Priority_Score":    score,
                    "Qty_Reason":        f"S{scenario_id}: split across 2 machines",
                    "Stagger_Adjusted":  "No",
                })
                print(f"  {part:<30}  {score:>6.1f}  {m:<15}  "
                      f"{run_h:>5.2f}  {qty:>8.0f}  "
                      f"{'Yes' if co_hrs>0 else 'No':>4}  "
                      f"✓ [{split_label}]")
            already_planned.add(part)
            continue

        # ── Truly not planned ─────────────────────────────────
        free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                    for m in compatible_mch if m in machine_hours}
        reason = ("Runner locked to specific machine — no capacity there"
                  if runner_lock else
                  "All compatible machines full — no trim/split possible")
        not_planned.append({
            "Part":                part,
            "Category":            category,
            "Score":               score,
            "Daily_Indent":        round(daily, 2),
            "Inventory_Now":       round(inv_now, 0),
            "Inv_Days_Coverage":   round(inv_days, 2),
            "Today_Target":        round(today_target_qty.get(part, 0), 0),
            "Compatible_Machines": ", ".join(compatible_mch),
            "Machine_Free_Hrs":    str(free_map),
            "Reason":              reason,
            "Action_Needed":       "Review matrix or add compatible machines",
        })
        print(f"  {part:<30}  {score:>6.1f}  {'—':<15}  {'—':>5}  "
              f"{'—':>8}  {'—':>4}  ✗ NOT PLANNED — {reason[:35]}")

    # ── 22H Utilization Enforcer ──────────────────────────────
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id,
        priority_scores)

    # ── CSP Tool-Changer ──────────────────────────────────────
    stagger_changeovers(plan, machines, changeover_dict)

    # ── Build output DataFrames ───────────────────────────────

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        r_val   = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        inv_b   = inventory.get(p, 0)
        gap     = daily - planned
        meets   = planned >= daily

        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Rate_Per_Hour":      round(r_val, 2),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(gap, 0),
            "Hrs_For_Daily":      round(daily / r_val if r_val > 0 else 0, 2),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
            "Qty_Reason":         row.get("Qty_Reason", "—"),
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(monthly, 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"      if days_cov >= 1
                                else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in machines:
        used       = machine_hours.get(m, 0)
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        co_count   = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        util_pct   = round(used / AVAILABLE_HOURS * 100, 1)
        util_status = ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                       "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                       "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                       "UNDERUSED")
        mach_rows.append({
            "Machine":             m,
            "Used_Hours":          round(used, 2),
            "Unused_Hours":        round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":       util_pct,
            "Status":              util_status,
            "Parts_Planned":       len(parts_run),
            "Changeovers":         co_count,
            "Last_Part_Run":       machine_last_part.get(m) or "—",
            "All_Parts_On_Machine":  ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df = pd.DataFrame(micro_idle) if micro_idle else pd.DataFrame()

    plan_df  = pd.DataFrame(plan)     if plan     else pd.DataFrame()
    def_df   = pd.DataFrame(deferred) if deferred else pd.DataFrame()
    not_df   = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df  = pd.DataFrame(mach_rows)
    inv_df   = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    # ── Summary ───────────────────────────────────────────────
    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY")
    print(f"    Scenario         : {scenario_desc}")
    print(f"    Parts planned    : {len(already_planned)}")
    print(f"    Not planned      : {len(not_planned)}")
    print(f"    Deferred         : {len(deferred)}")
    if not mach_df.empty:
        avg_util = mach_df["Utilization_%"].mean()
        full_cnt = (mach_df["Status"] == "FULL").sum()
        good_cnt = (mach_df["Status"] == "GOOD").sum()
        under_cnt = (mach_df["Status"] == "UNDERUSED").sum()
        print(f"    Avg utilization : {avg_util:.1f}%")
        print(f"    FULL machines   : {full_cnt}")
        print(f"    GOOD machines   : {good_cnt}")
        print(f"    UNDERUSED       : {under_cnt}")
    if micro_idle:
        print(f"    Micro-idle      : {len(micro_idle)} machines  "
              f"(all eligible parts exhausted)")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df)

# =============================================================
# SECTION 20 — RUN SCHEDULER
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

# Part audit (same as V6)
all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"

    if part in zero_rate_set or r_val is None:
        status = "ZERO/MISSING CYCLE TIME"
        gate   = "GATE 1"
        reason = f"Cycle time={ct_raw} — missing or zero"
    elif part not in matrix_parts:
        status = "NOT IN VT_MATRIX"
        gate   = "GATE 2"
        reason = "No compatible machine assigned"
    elif monthly == 0:
        status = "ZERO/MISSING INDENT"
        gate   = "GATE 3"
        reason = "Monthly indent is zero"
    elif should_skip(part)[0]:
        status = "SKIPPED (LOW INDENT / TRIVIAL RUN)"
        gate   = "GATE 4"
        reason = should_skip(part)[1]
    elif daily > 0 and inv >= daily:
        status = "NOT REQUIRED — INV SUFFICIENT"
        gate   = "GATE 5"
        reason = f"Inventory ({inv:.0f}) ≥ daily_indent ({daily:.2f})"
    else:
        status = "ENTERS SCHEDULER"
        gate   = "—"
        reason = "Passed all gates"

    audit_rows.append({
        "Part":          part,
        "Gate_Failed":   gate,
        "Reason":        reason,
        "Monthly_Indent":round(monthly, 0),
        "Daily_Indent":  round(daily, 2),
        "Inventory":     round(inv, 0),
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Cycle_Time":    ct_raw,
        "Cavity":        cv_raw,
        "Status":        status,
    })

audit_df = pd.DataFrame(audit_rows)

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro_idle) = schedule(
    vt_parts, vt_compat, vt_machines, vt_changeover, "VT Machines"
)

save_machine_state(vt_state)

# =============================================================
# SECTION 21 — BUILD MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df, machines):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        cumulative = 0.0
        seq = 1
        for _, pr in machine_rows.iterrows():
            run_h   = sf(pr.get("Run_Hours", 0))
            co_h    = sf(pr.get("Changeover_Hrs", 0))
            r_val   = sf(pr.get("Rate_Per_Hour", 0))
            start_h = cumulative + co_h
            end_h   = start_h + run_h

            rows.append({
                "Machine":               m,
                "Seq":                   seq,
                "Part":                  pr.get("Part", "—"),
                "Category":              part_category.get(pr.get("Part", ""), "Stranger"),
                "Priority_Score":        round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":         round(r_val, 2),
                "Changeover_Before_Mins":round(co_h * 60, 1),
                "Run_Hours":             round(run_h, 2),
                "Start_Time":            _fmt_h(cumulative),
                "Start_After_CO":        _fmt_h(start_h),
                "End_Time":              _fmt_h(end_h),
                "Cumulative_Hrs":        round(end_h, 2),
                "Production_Qty":        sf(pr.get("Production_Qty", 0)),
                "Monthly_Indent":        sf(pr.get("Monthly_Indent", 0)),
                "Daily_Indent":          sf(pr.get("Daily_Indent", 0)),
                "Today_Target":          sf(pr.get("Today_Target", 0)),
                "Changeover":            pr.get("Changeover", "No") or "No",
                "Type":                  pr.get("Type", "Primary") or "Primary",
                "Qty_Reason":            pr.get("Qty_Reason", "—"),
                "Row_Type":              "Part",
            })
            cumulative = end_h
            seq += 1

        total_used  = round(cumulative, 2)
        total_qty   = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_total    = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        co_count    = int(machine_rows["Changeover"].eq("Yes").sum())

        rows.append({
            "Machine":               m,
            "Seq":                   "—",
            "Part":                  f"TOTAL — {m}",
            "Category":              "—",
            "Priority_Score":        "—",
            "Rate_Per_Hour":         "—",
            "Changeover_Before_Mins":round(co_total * 60, 1),
            "Run_Hours":             round(total_used - co_total, 2),
            "Start_Time":            "00:00",
            "Start_After_CO":        "—",
            "End_Time":              _fmt_h(total_used),
            "Cumulative_Hrs":        total_used,
            "Production_Qty":        round(total_qty, 0),
            "Monthly_Indent":        "—",
            "Daily_Indent":          "—",
            "Today_Target":          "—",
            "Changeover":            f"{co_count} changeovers",
            "Type":                  f"Used {total_used}h / {AVAILABLE_HOURS}h  |  "
                                     f"Unused {round(AVAILABLE_HOURS-total_used,2)}h  |  "
                                     f"Util {round(total_used/AVAILABLE_HOURS*100,1)}%",
            "Qty_Reason":            "—",
            "Row_Type":              "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)

# =============================================================
# SECTION 22 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_Plan":                  "1F4E79",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Deferred":              "7F6000",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
    "VT_Daily_Indent_Status":   "0F4C2A",
    "VT_Priority_Scores":       "2C4770",
    "VT_Micro_Idle":            "5C3D2E",
}

STATUS_FILLS = {
    "FULL":     PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":     PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":  PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":PatternFill("solid", fgColor="FFC7CE"),
    "OK":       PatternFill("solid", fgColor="C6EFCE"),
    "LOW":      PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL": PatternFill("solid", fgColor="FFC7CE"),
    "YES ✓":    PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":     PatternFill("solid", fgColor="FFC7CE"),
    "PRODUCTION NEEDED":        PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":        PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":           PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                  PatternFill("solid", fgColor="EDEDED"),
    "NO INDENT":                PatternFill("solid", fgColor="EDEDED"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                  PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily", "Covers_With"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_font  = Font(bold=True, color="FFFFFF", size=11)
    summary_fill = PatternFill("solid", fgColor="0D9488")
    summary_font = Font(bold=True, color="FFFFFF", size=11)
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None

    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = summary_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            bg = part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = bg
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col - 1].value == "Yes":
                row[co_col - 1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "C2"


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw = build_machine_wise_plan(vt_plan, vt_machines)

sheets = {
    "VT_Plan_By_Machine":     vt_mw,
    "VT_Plan":                vt_plan,
    "VT_Daily_Indent_Status": vt_indent_status,
    "VT_Priority_Scores":     vt_scores,
    "VT_Machine_Util":        vt_mach,
    "VT_Not_Planned":         vt_not,
    "VT_Deferred":            vt_def,
    "VT_Inventory_Health":    vt_inv,
    "VT_Indent_Horizon":      vt_horizon,
    "VT_Part_Audit":          audit_df,
}
if not vt_micro_idle.empty:
    sheets["VT_Micro_Idle"] = vt_micro_idle

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name != "VT_Plan_By_Machine":
        style_sheet(wb[sheet_name], header_hex)

# Colour the two YES/NO columns in Daily_Indent_Status
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    miss_fill  = PatternFill("solid", fgColor="FFC7CE")
    meet_fill  = PatternFill("solid", fgColor="C6EFCE")
    gap_fill   = PatternFill("solid", fgColor="FFEB9C")
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = meet_fill if str(cell.value) == "YES ✓" else miss_fill
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = meet_fill if str(cell.value) == "YES ✓" else gap_fill
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

tab_colors = {k: v for k, v in HEADER_COLORS.items()}
for name, color in tab_colors.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 23 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V7 Complete  —  {PLANNING_DATE}")
print(f"  Level 2 Intelligence: Priority Scoring + Cross-Machine Routing")
print(f"{'='*65}")

gate_counts = audit_df["Status"].value_counts()
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<45}: {count:>4}")

print(f"\n  Scheduling result:")
print(f"    Planned        : {len(vt_plan):>4}")
print(f"    Not planned    : {len(vt_not):>4}  (see VT_Not_Planned sheet)")
print(f"    Deferred       : {len(vt_def):>4}  (inv sufficient today)")

if not vt_mach.empty:
    avg_u  = vt_mach["Utilization_%"].mean()
    max_u  = vt_mach["Utilization_%"].max()
    min_u  = vt_mach["Utilization_%"].min()
    under  = (vt_mach["Utilization_%"] < UTIL_TARGET_PCT).sum()
    print(f"\n  Machine utilization:")
    print(f"    Average     : {avg_u:.1f}%")
    print(f"    Best        : {max_u:.1f}%")
    print(f"    Worst       : {min_u:.1f}%")
    print(f"    Below {UTIL_TARGET_PCT}%  : {under} machines  "
          f"(all eligible parts already planned)")

if not vt_indent_status.empty:
    meets  = (vt_indent_status["Meets_Daily_Indent"] == "YES ✓").sum()
    misses = (vt_indent_status["Meets_Daily_Indent"] == "NO ✗").sum()
    covers = (vt_indent_status.get("Covers_With_Inv", pd.Series()) == "YES ✓").sum()
    print(f"\n  Daily indent coverage:")
    print(f"    Production meets indent    : {meets} parts  ✓")
    print(f"    Production misses indent   : {misses} parts  "
          f"(see VT_Daily_Indent_Status)")
    print(f"    Covered incl. inventory    : {covers} parts")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date({PLANNING_DATE.year}, "
      f"{PLANNING_DATE.month}, {PLANNING_DATE.day + 1})")
print(f"{'='*65}")

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling
#
#  What's new vs V7:
#  ─────────────────────────────────────────────────────────────
#  1. TOOL-AWARE SCHEDULING
#     Every part has a tool count (column "Tools" in VT sheet).
#     Tools control how many machines can run the part in parallel.
#
#     Phase 1 — Primary machine
#       Assign best-ranked compatible machine.
#       Run for min(available_hours, hours_to_meet_daily_indent).
#       If daily indent fully met → go to Phase 3.
#
#     Phase 2 — Tool expansion (only if Phase 1 fell short)
#       shortfall = daily_indent − qty_produced_on_machine_1
#       If tools ≥ 2 and another compatible machine has capacity:
#         Assign shortfall-only hours to machine 2.
#       If still short and tools ≥ 3 → machine 3, etc.
#       Max parallel machines = min(tools, compatible machines).
#       Second machine starts mid-shift (not from 00:00) — it only
#       fills what machine 1 couldn't.
#
#     Phase 3 — Inventory build (after daily indent fully met)
#       Extend the SAME machine(s) already running the part.
#       Cap = scenario OPD. No new tools consumed for inventory.
#
#  2. TWO NEW EXCEL OUTPUT SHEETS
#
#     VT_Multi_Machine_Parts
#       Only parts running on ≥ 2 machines.
#       Columns: Part | Machine | Run_Hours | Production_Qty |
#                Role (Primary / Tool-Expansion) | Tools_Available |
#                Machines_Used | Total_Qty_Across_Machines
#
#     VT_Production_vs_Indent
#       Every planned part.
#       Columns: Part | Machine(s) | Total_Qty_Produced |
#                Daily_Indent | Gap (+over / −under) |
#                Gap_Direction (OVER / UNDER / MET) |
#                Extra_Days_Stock (if over) |
#                Inventory_Before | Inventory_After
#
#  Planning logic (unchanged):
#    daily_indent  = monthly_indent / working_days
#    Rate          = 3600 / cycle_time  (cavities cancel)
#    working_days  = calendar days − Sundays
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
TARGET_DAYS_INV      = 3
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 98.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V8_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns
                  if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

# Rate = 3600 / cycle_time  (cavities cancel — full cavity operation assumed)
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

# Tools available per part (default 1 if missing)
tools_available = {}
for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    return False, ""

# =============================================================
# SECTION 7B — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7C — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  ({len(state)} machines with history)")
        return state
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{TARGET_DAYS_INV} days)"

# =============================================================
# SECTION 11 — OPD CAP BY SCENARIO
# =============================================================

def opd_cap(scenario_id):
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, TARGET_DAYS_INV - days_cov) / TARGET_DAYS_INV)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if (last is None or last == part) else \
                 vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        co_penalty      = co_hrs / AVAILABLE_HOURS
        util_penalty    = used / AVAILABLE_HOURS
        same_part_bonus = -0.20 if (last == part) else 0.0
        cost = co_penalty + util_penalty + same_part_bonus
        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — TOOL-AWARE ASSIGNMENT  (core new logic in V8)
# =============================================================
#
# assign_part() handles all three phases for a single part:
#
#   Phase 1 — Primary machine
#     Run on best machine for min(effective_free, hrs_for_daily_indent).
#     If produced_qty ≥ daily_indent → skip Phase 2.
#
#   Phase 2 — Tool expansion
#     shortfall_qty = daily_indent − produced_so_far
#     For each additional tool available (up to tools_available):
#       Find next best compatible machine with capacity.
#       Run only enough hours to cover the shortfall.
#       Stop as soon as shortfall is fully covered.
#
#   Phase 3 — Inventory build
#     After daily indent met, extend machine(s) already running
#     the part up to the OPD cap. No new machines assigned.
#
# Returns list of plan rows added, or empty list if not planned.
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    """
    Assigns a part across 1–N machines using tool-aware logic.
    Returns list of new plan rows (empty = not planned).
    """
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0

    # ── PHASE 1: Primary machine ──────────────────────────────
    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days)

    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    # Hours needed to meet daily indent from current inventory
    shortfall_qty  = max(0.0, daily - inv_now)
    hrs_for_indent = shortfall_qty / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_indent = max(MIN_RUN_HOURS, hrs_for_indent)

    # Phase 1 runs for: min(effective_free, hrs_for_indent)
    run1 = min(eff1, hrs_for_indent)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far         += qty1
    tools_used              += 1

    new_rows.append({
        "Part":             part,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })

    already_planned.add(part)

    # ── PHASE 2: Tool expansion (only if daily indent not yet met) ──
    if produced_so_far < daily and tools > 1:
        # Machines already used by this part in this assignment
        used_machines = {m1}

        while produced_so_far < daily and tools_used < tools:
            shortfall_now = daily - produced_so_far
            hrs_needed    = shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS

            # Candidate machines: compatible, not already used for this part
            remaining_machines = [
                m for m in compatible if m not in used_machines
            ]
            ranked2, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days)

            if not ranked2:
                break   # no more machines available

            m2, co2, eff2, _ = ranked2[0]

            # Only run enough to cover the shortfall
            run2 = min(eff2, max(MIN_RUN_HOURS, hrs_needed))
            qty2 = round(run2 * r_val, 0)

            machine_hours[m2]       = round(machine_hours.get(m2, 0) + co2 + run2, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty2, 0)
            machine_last_part[m2]   = part
            produced_so_far        += qty2
            tools_used             += 1
            used_machines.add(m2)

            new_rows.append({
                "Part":             part,
                "Category":         category,
                "Machine":          m2,
                "Run_Hours":        round(run2, 3),
                "Changeover_Hrs":   round(co2, 3),
                "Total_Hrs_Used":   round(co2 + run2, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty2,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if co2 == 0 else "Yes",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Stagger_Adjusted": "No",
            })

            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tools} → "
                  f"{m2:15s}  {run2:.2f}h  qty={qty2:.0f}  "
                  f"(shortfall was {shortfall_now:.0f})")

    # ── PHASE 3: Inventory build on same machines ─────────────
    # Extend existing run rows for this part up to OPD cap.
    # No new machines, no new tools.
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m       = row["Machine"]
            free_m  = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

    # Update Tools_Used on all rows for this part
    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# SECTION 15 — CSP TOOL-CHANGER  (unchanged from V7)
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra

def _csp_order_events(events):
    n = len(events)
    if n == 0:
        return events
    try:
        from constraint import Problem
        SLOTS = int(AVAILABLE_HOURS * 60)
        problem = Problem()
        for i, ev in enumerate(events):
            dur_min = int(math.ceil(ev["co_duration"] * 60))
            problem.addVariable(i, range(0, max(0, SLOTS - dur_min) + 1))
        for i in range(n):
            for j in range(i+1, n):
                di = int(math.ceil(events[i]["co_duration"] * 60))
                dj = int(math.ceil(events[j]["co_duration"] * 60))
                def make_c(di, dj):
                    return lambda si, sj: (si+di <= sj) or (sj+dj <= si)
                problem.addConstraint(make_c(di, dj), (i, j))
        solutions = problem.getSolutions()
        if not solutions:
            return sorted(events, key=lambda e: e["natural_start"])
        best    = min(solutions, key=lambda sol: sum(sol.values()))
        ordered = sorted(range(n), key=lambda i: best[i])
        return [events[i] for i in ordered]
    except Exception:
        return sorted(events, key=lambda e: e["natural_start"])

def stagger_changeovers(plan, machines):
    print(f"\n  CSP Tool-Changer Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print("  No changeovers — tool changer idle  ✓")
        return

    ordered_events       = _csp_order_events(events)
    tool_changer_free_at = 0.0
    total_extra_pcs      = 0

    print(f"\n  {'#':<4} {'Machine':<18} {'Part Before':<24} {'Part After':<24} "
          f"{'CO':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7}")
    print(f"  {'─'*100}")

    for idx, ev in enumerate(ordered_events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = actual_start - natural_start

        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs

        ev["actual_start"]   = actual_start
        tool_changer_free_at = actual_start + co_h
        wait_str = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<24} "
              f"{ev['part_after']:<24} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7}")

    print(f"\n  Tool-changer finishes at {_fmt_h(tool_changer_free_at)}  |  "
          f"Extra pcs from wait-fill: {total_extra_pcs:,.0f}")

# =============================================================
# SECTION 16 — 22H UTILIZATION ENFORCER
# =============================================================

def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER  (target ≥{UTIL_TARGET_PCT}%)")
    micro_idle_log = []

    machines_by_util = sorted(
        vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # STEP 1: Extend existing parts on this machine
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    ↑ EXTEND {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        # STEP 2: Add new parts
        if remaining < MIN_RUN_HOURS:
            continue

        candidates = []
        for p in all_parts:
            if p in already_planned:
                continue
            if m not in vt_compat.get(p, []):
                continue
            skip, _ = should_skip(p)
            if skip:
                continue
            daily_p = indent_daily.get(p, 0)
            inv_now = current_inventory.get(p, 0)
            if inv_now >= opd_cap(scenario_id) * daily_p:
                continue
            candidates.append(p)

        candidates.sort(key=lambda x: priority_scores.get(x, 0), reverse=True)

        for p in candidates:
            if remaining < MIN_RUN_HOURS:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            inv_days = inv_now / daily_p if daily_p > 0 else 999
            last     = machine_last_part.get(m)
            co_hrs   = 0.0 if (last is None or last == p) else \
                       vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue

            # Run to meet daily indent shortfall first, then cap
            shortfall = max(0.0, daily_p - inv_now)
            hrs_for_indent = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            run_hrs  = min(eff_free, max(hrs_for_indent,
                                         headroom / r_val if r_val > 0 else eff_free))
            run_hrs  = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            qty      = round(run_hrs * r_val, 0)

            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            already_planned.add(p)
            remaining = round(remaining - co_hrs - run_hrs, 4)

            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "Filler (utilization enforcer)",
                "Role":             "Primary",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       1,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            1,
                "Stagger_Adjusted": "No",
            })
            print(f"    + ADD  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  "
                  f"[{'CO' if co_hrs>0 else 'No CO'}]")

        # STEP 3: Log micro-idle
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            "All compatible parts planned or at OPD cap",
                })
                print(f"    ⚠  {m:15s}  idle {final_remaining:.2f}h ({util_final}%)")

    return micro_idle_log

# =============================================================
# SECTION 17 — NEW OUTPUT: MULTI-MACHINE PARTS VIEW
# =============================================================

def build_multi_machine_view(plan):
    """
    Returns a DataFrame showing only parts that were assigned
    to more than one machine, with one row per machine assignment.
    Includes a summary row per part showing totals.
    """
    if not plan:
        return pd.DataFrame()

    # Group plan rows by part
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)

    # Keep only parts with multiple machine assignments
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}

    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(),
                              key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily  = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)

        for row in rows:
            qty_this   = float(row["Production_Qty"])
            run_h      = float(row["Run_Hours"])
            output_rows.append({
                "Part":                  part,
                "Category":              part_category.get(part, "Stranger"),
                "Tools_Available":       tools_available.get(part, 1),
                "Machines_Used":         len(rows),
                "Machine":               row["Machine"],
                "Role":                  row.get("Role", "Primary"),
                "Run_Hours":             round(run_h, 2),
                "Changeover_Hrs":        round(float(row.get("Changeover_Hrs", 0)), 2),
                "Production_Qty":        round(qty_this, 0),
                "Daily_Indent":          round(daily, 2),
                "Total_Qty_All_Machines":round(total_qty, 0),
                "Type":                  row.get("Type", "—"),
            })

        # Summary row for this part
        output_rows.append({
            "Part":                  f"  ↳ TOTAL — {part}",
            "Category":              "—",
            "Tools_Available":       tools_available.get(part, 1),
            "Machines_Used":         len(rows),
            "Machine":               f"{len(rows)} machines",
            "Role":                  "TOTAL",
            "Run_Hours":             round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":        round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          round(daily, 2),
            "Total_Qty_All_Machines":round(total_qty, 0),
            "Type":                  "—",
        })
        # Blank spacer
        output_rows.append({k: "" for k in output_rows[-1].keys()})

    return pd.DataFrame(output_rows)

# =============================================================
# SECTION 18 — NEW OUTPUT: PRODUCTION VS INDENT VIEW
# =============================================================

def build_production_vs_indent(plan, all_parts):
    """
    Every planned part: one row showing total production across
    all machines vs daily indent, gap direction, and extra days stock.
    """
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_qty     = defaultdict(float)
    part_machines = defaultdict(list)

    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)   # +ve = over, -ve = under
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")

        # Extra days stock = how many days the produced qty covers
        # beyond the daily indent
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0

        machines_str = ", ".join(dict.fromkeys(part_machines[p]))  # unique, ordered

        rows.append({
            "Part":                  p,
            "Category":              part_category.get(p, "Stranger"),
            "Tools_Available":       tools_available.get(p, 1),
            "Machines":              machines_str,
            "Machines_Count":        len(set(part_machines[p])),
            "Total_Qty_Produced":    produced,
            "Daily_Indent":          round(daily, 2),
            "Monthly_Indent":        round(monthly, 0),
            "Gap_vs_Daily":          gap,      # +ve = over, -ve = under
            "Gap_Direction":         gap_dir,
            "Extra_Days_Stock":      extra_days,
            "Inventory_Before":      round(inv_b, 0),
            "Inventory_After":       inv_after,
            "Days_Coverage_After":   round(inv_after / daily, 2) if daily > 0 else 0,
        })

    # Sort: UNDER first (most urgent), then MET, then OVER
    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"])
        df = df.reset_index(drop=True)
    return df

# =============================================================
# SECTION 19 — INDENT HORIZON TABLE
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 20 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # Active parts
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and not (inventory.get(p, 0) >= indent_daily.get(p, 0) > 0)
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    # Sort by score descending
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0), reverse=True)

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts, score order)")
    print(f"  {'Part':<30} {'Score':>6} {'Tools':>5} {'Machine(s)':<30} "
          f"{'Run':>5} {'Qty':>8}  Status")
    print(f"  {'─'*95}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")

        if monthly == 0:
            deferred.append({"Part": part, "Category": category,
                             "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  DEFERRED (no indent)")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI-MACHINE]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {machines_str:<30}  "
                  f"{total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            inv_days = inv_now / daily if daily > 0 else 999
            free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                        for m in vt_compat.get(part, []) if m in machine_hours}
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Today_Target":        round(today_target_qty.get(part, 0), 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Machine_Free_Hrs":    str(free_map),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NO CAPACITY")

    # 22H Utilization Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores)

    # CSP Tool-Changer
    stagger_changeovers(plan, vt_machines)

    # ── Build output views ────────────────────────────────────
    multi_machine_df     = build_multi_machine_view(plan)
    prod_vs_indent_df    = build_production_vs_indent(plan, list(parts))

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        r_val   = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"  if days_cov >= 1
                                else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m
                        and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df = pd.DataFrame(micro_idle)     if micro_idle else pd.DataFrame()
    plan_df  = pd.DataFrame(plan)           if plan       else pd.DataFrame()
    def_df   = pd.DataFrame(deferred)       if deferred   else pd.DataFrame()
    not_df   = pd.DataFrame(not_planned)    if not_planned else pd.DataFrame()
    mach_df  = pd.DataFrame(mach_rows)
    inv_df   = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    # Summary
    multi_count = len(multi_machine_df[
        multi_machine_df.get("Role", pd.Series()) == "TOTAL"
    ]) if not multi_machine_df.empty else 0

    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned          : {len(already_planned)}")
    print(f"    Parts on multi-machine : {multi_count}")
    print(f"    Not planned            : {len(not_planned)}")
    print(f"    Deferred               : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization        : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    UNDERUSED machines     : "
              f"{(mach_df['Status']=='UNDERUSED').sum()}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df)

# =============================================================
# SECTION 21 — PART AUDIT
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif should_skip(part)[0]:
        status, gate, reason = "SKIPPED (LOW INDENT / TRIVIAL RUN)", "GATE 4", should_skip(part)[1]
    elif daily > 0 and inv >= daily:
        status, gate, reason = "NOT REQUIRED — INV SUFFICIENT", "GATE 5", \
            f"Inventory ({inv:.0f}) ≥ daily_indent ({daily:.2f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":          part,
        "Gate_Failed":   gate,
        "Reason":        reason,
        "Monthly_Indent":round(monthly, 0),
        "Daily_Indent":  round(daily, 2),
        "Inventory":     round(inv, 0),
        "Tools":         tools,
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Cycle_Time":    ct_raw,
        "Cavity":        cv_raw,
        "Status":        status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<45}: {count:>4}")

# =============================================================
# SECTION 22 — RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 23 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        cumulative, seq = 0.0, 1
        for _, pr in machine_rows.iterrows():
            run_h = sf(pr.get("Run_Hours", 0))
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            start_h = cumulative + co_h
            end_h   = start_h + run_h
            rows.append({
                "Machine":               m,
                "Seq":                   seq,
                "Part":                  pr.get("Part", "—"),
                "Category":              part_category.get(pr.get("Part",""), "Stranger"),
                "Role":                  pr.get("Role", "Primary"),
                "Tools_Available":       pr.get("Tools_Available", 1),
                "Priority_Score":        round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":         round(r_val, 2),
                "Changeover_Before_Mins":round(co_h * 60, 1),
                "Run_Hours":             round(run_h, 2),
                "Start_Time":            _fmt_h(cumulative),
                "Start_After_CO":        _fmt_h(start_h),
                "End_Time":              _fmt_h(end_h),
                "Cumulative_Hrs":        round(end_h, 2),
                "Production_Qty":        sf(pr.get("Production_Qty", 0)),
                "Daily_Indent":          sf(pr.get("Daily_Indent", 0)),
                "Today_Target":          sf(pr.get("Today_Target", 0)),
                "Changeover":            pr.get("Changeover", "No") or "No",
                "Type":                  pr.get("Type", "Primary") or "Primary",
                "Row_Type":              "Part",
            })
            cumulative = end_h
            seq += 1

        total_used = round(cumulative, 2)
        co_total   = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        total_qty  = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count   = int(machine_rows["Changeover"].eq("Yes").sum())
        rows.append({
            "Machine":               m,
            "Seq":                   "—",
            "Part":                  f"TOTAL — {m}",
            "Category":              "—",
            "Role":                  "—",
            "Tools_Available":       "—",
            "Priority_Score":        "—",
            "Rate_Per_Hour":         "—",
            "Changeover_Before_Mins":round(co_total * 60, 1),
            "Run_Hours":             round(total_used - co_total, 2),
            "Start_Time":            "00:00",
            "Start_After_CO":        "—",
            "End_Time":              _fmt_h(total_used),
            "Cumulative_Hrs":        total_used,
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          "—",
            "Today_Target":          "—",
            "Changeover":            f"{co_count} changeovers",
            "Type":                  (f"Used {total_used}h / {AVAILABLE_HOURS}h  |  "
                                      f"Unused {round(AVAILABLE_HOURS-total_used,2)}h  |  "
                                      f"Util {round(total_used/AVAILABLE_HOURS*100,1)}%"),
            "Row_Type":              "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)

# =============================================================
# SECTION 24 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
}

STATUS_FILLS = {
    "FULL":     PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":     PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":  PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":PatternFill("solid", fgColor="FFC7CE"),
    "OK":       PatternFill("solid", fgColor="C6EFCE"),
    "LOW":      PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL": PatternFill("solid", fgColor="FFC7CE"),
    "YES ✓":    PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":     PatternFill("solid", fgColor="FFC7CE"),
    "OVER":     PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":    PatternFill("solid", fgColor="FFC7CE"),
    "MET":      PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":        PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":        PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":           PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                  PatternFill("solid", fgColor="EDEDED"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                  PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col-1].value == "Yes":
                row[co_col-1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "C2"


def style_prod_vs_indent_sheet(ws):
    """Special styling for VT_Production_vs_Indent sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])

    headers   = [c.value for c in ws[1]]
    gap_col   = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    gap_v_col = headers.index("Gap_vs_Daily")  + 1 if "Gap_vs_Daily"  in headers else None

    over_fill  = PatternFill("solid", fgColor="DDEBF7")   # blue — over-produced
    under_fill = PatternFill("solid", fgColor="FFC7CE")   # red  — under-produced
    met_fill   = PatternFill("solid", fgColor="C6EFCE")   # green — exactly met

    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    """Special styling for VT_Multi_Machine_Parts sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])

    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None

    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")

    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw = build_machine_wise_plan(vt_plan)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_Plan":                 vt_plan,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames:
    style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts" in wb.sheetnames:
    style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent", "VT_Multi_Machine_Parts"):
        style_sheet(wb[sheet_name], header_hex)

# Colour Daily Indent Status YES/NO columns
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 25 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V8 Complete  —  {PLANNING_DATE}")
print(f"  Tool-Aware Multi-Machine Scheduling")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<45}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_multi_machine.empty:
    n_multi = vt_multi_machine[vt_multi_machine["Role"] == "TOTAL"].shape[0]
    print(f"    Multi-machine parts: {n_multi:>4}  (see VT_Multi_Machine_Parts)")

if not vt_prod_vs_indent.empty:
    over  = (vt_prod_vs_indent["Gap_Direction"] == "OVER").sum()
    under = (vt_prod_vs_indent["Gap_Direction"] == "UNDER").sum()
    met   = (vt_prod_vs_indent["Gap_Direction"] == "MET").sum()
    print(f"\n  Production vs Daily Indent:")
    print(f"    Over-produced  : {over:>4} parts  (inv building — see VT_Production_vs_Indent)")
    print(f"    Under-produced : {under:>4} parts  (machine capacity limited)")
    print(f"    Exactly met    : {met:>4} parts")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(vt_mach['Status']=='UNDERUSED').sum()} machines")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")